# Evolve your harness

One full pass of the harness evolution loop, cell by cell: serve Reef,
send three coding tasks through it, report each score, and watch one
gated evolve step improve a skill and publish a new harness version.

On Colab, pick a GPU runtime (T4 is enough) and run the setup cell below;
it installs everything and serves the model. A local run instead needs:
`pip install reef-client`, the `pi` binary on PATH
(`npm i -g @earendil-works/pi-coding-agent@0.84.2`), and an
OpenAI-compatible endpoint serving a model. Set the three values in the
next cell. No GPU is needed by Reef itself.

[<img align="left" src="https://colab.research.google.com/assets/colab-badge.svg">](https://colab.research.google.com/github/Human-Agent-Society/reef/blob/main/tutorials/evolve-your-harness/evolve-your-harness.ipynb)

In [ ]:
# Colab setup: clone the repo, install Reef and the pi agent, and serve a
# model with ollama (pick a GPU runtime; T4 is enough). A run from a local
# checkout skips this cell.
import importlib.util
import os
import pathlib
import subprocess
import sys

if importlib.util.find_spec("google.colab") is not None:
    root = pathlib.Path("/content/reef")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Human-Agent-Society/reef", str(root)], check=True)
    os.chdir(root / "tutorials" / "evolve-your-harness")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
    subprocess.run(["npm", "install", "-g", "--silent",
                    "@earendil-works/pi-coding-agent@0.84.2"], check=True)
    if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
        # A pinned release asset, extracted directly: install.sh fails in a
        # container (its service-manager setup has nothing to manage), and the
        # ollama.com tarball alias 404s upstream since the release assets
        # moved to zstd.
        subprocess.run(["apt-get", "update", "-qy"], check=True)
        subprocess.run(["apt-get", "install", "-qy", "zstd"], check=True)
        subprocess.run("curl -fSL https://github.com/ollama/ollama/releases/download/"
                       "v0.33.2/ollama-linux-amd64.tar.zst"
                       " | tar --zstd -x -C /usr/local", shell=True, check=True)
    subprocess.run("nohup ollama serve >/tmp/ollama.log 2>&1 &", shell=True, check=True)
    import time
    import urllib.request
    for _ in range(30):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434", timeout=1).close()
            break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start within 30s; see /tmp/ollama.log")
    subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)
    print("colab setup complete")


In [1]:
# The model under test: any OpenAI-compatible endpoint (no /v1 suffix).
UPSTREAM_URL = "http://127.0.0.1:11434"
UPSTREAM_MODEL = "qwen2.5:7b"
UPSTREAM_API_KEY = "sk-local"


## Start Reef

`serve.yaml` carries the whole stack: the service and the
`harness_evolve` recipe sections (seed skill composition, the three
evolve tasks, the gate). We patch the upstream binding AND the recipe's
model binding to the values above (the recipe's `model.path` names the
model under test: the proposer and the evaluation episodes call it, so a
mismatch leaves the evolve step without a working model), materialize
the recipe config from the patched text, and start the service as a
subprocess. Reef is a plain Python process; the model stays wherever
it is served.

In [2]:
import json
import os
import pathlib
import re
import subprocess
import time
import urllib.request

import yaml

here = pathlib.Path.cwd()
work = here / "work"
(work / "recipes").mkdir(parents=True, exist_ok=True)

text = (here / "configs" / "serve.yaml").read_text()
text = re.sub(r"upstream_url: .*", f"upstream_url: {UPSTREAM_URL}", text)
text = re.sub(r"upstream_model: .*", f"upstream_model: {UPSTREAM_MODEL}", text)
text = re.sub(r"(?m)^  path: .*", f"  path: {UPSTREAM_MODEL}", text)
(work / "serve-notebook.yaml").write_text(text)

env = dict(os.environ)
env["REEF_RECIPE_CONFIG_DIR"] = str(work / "recipes")
env["REEF_UPSTREAM_API_KEY"] = UPSTREAM_API_KEY
env["PYTHONPATH"] = str(here.parent.parent)

# What materialize_recipe.py does, from the patched text instead of serve.yaml.
cfg = yaml.safe_load(text)
recipe = {key: cfg[key] for key in ("implementation", "model", "evolution", "data")}
(work / "recipes" / "harness_evolve.yaml").write_text(yaml.safe_dump(recipe, sort_keys=False))
(work / "tasks.json").write_text(json.dumps(recipe["evolution"]["tasks"], indent=2) + "\n")

serve = subprocess.Popen(
    ["python3", "-m", "reef", "serve", "-c", str(work / "serve-notebook.yaml")],
    env=env, stdout=(work / "reef.log").open("w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8900/healthz", timeout=1)
        break
    except Exception:
        assert serve.poll() is None, (work / "reef.log").read_text()[-2000:]
        time.sleep(1)
print("reef is up")


reef is up


## Send the tasks and report the scores

Each task goes through Reef's inference proxy, which injects the served
skill catalog and records the exchange. The report ties the score to
that recorded traffic. Only failing reports batch; the first failure
schedules one evolve step.

In [3]:
import json
from reef_client import ReefClient
from harness import evolution

SCENARIO = "harness-evolve-demo"
client = ReefClient("http://127.0.0.1:8900", token="reef-local", timeout_s=300.0)
tasks = json.loads((work / "tasks.json").read_text())

failures = 0
for index, task in enumerate(tasks, start=1):
    body, receipt = client.inference_with_record(
        SCENARIO, "/v1/chat/completions",
        {"model": UPSTREAM_MODEL, "messages": [{"role": "user", "content": task}]},
    )
    prefix = task.split(maxsplit=1)[0]
    score = evolution.grade_text(task, body["choices"][0]["message"]["content"])
    client.report(
        SCENARIO,
        {"agent_record_id": f"harness-evolve-{index}", "score": score, "feedback": prefix},
        references=[receipt],
    )
    failures += score == 0.0
    print(f"task {index} {prefix}: score {score}")
print(f"{failures} failing report(s); each schedules one gated evolve step"
      if failures else "every task passed: nothing batches, no evolve step runs")


task 1 [sieve]: score 1.0


task 2 [fib]: score 0.0


task 3 [csv]: score 1.0
1 failing report(s); each schedules one gated evolve step


## Watch the evolve step publish

The step proposes one skill mutation (the served model is its own
proposer), renders candidate and current compositions, runs the three
tasks twice each as headless `pi` episodes, and publishes only if the
candidate wins. `GET /reef/harness` answers 404 until then.

In [4]:
from reef_client import ReefClientError

manifest = None
deadline = time.monotonic() + 900
while manifest is None and time.monotonic() < deadline and failures:
    try:
        manifest = client.get("/reef/harness", extra_headers={"x-reef-scenario": SCENARIO})
    except ReefClientError as exc:  # noqa: PERF203 - publish poll
        if exc.status != 404:
            raise
        if error := client.get("/reef/status").get("error"):
            raise SystemExit(f"evolve step failed: {error}") from exc
        time.sleep(2)
if manifest:
    print(f"published: artifact {manifest['release_id']}"
          f" (parent {manifest['parent_release_id']})")
    print(json.dumps(manifest["gate"], indent=2, sort_keys=True))
    for path, text in sorted(manifest["files"].items()):
        if "/skills/" in path:
            print(f"--- {path} ---\n{text}")
elif failures:
    # The commit log says why: a "skipped" metric means the proposer
    # returned no usable mutation; a selection reason means the candidate
    # ran its episodes and lost the gate.
    commits = [json.loads(line)
               for path in (work / "agent-record").glob("*.commits.jsonl")
               for line in path.read_text().splitlines()]
    metrics = commits[-1]["metrics"] if commits else {}
    why = metrics.get("skipped") or metrics.get("selection", {}).get("reason") or "no evolve step committed"
    print(f"no publish this time ({why}); rerun the task cell for another attempt")


published: artifact f98266d26785a02bb825b242b1911e3b168d0798 (parent bf022b0e8abc11ee71c58a693dbefe4f1ddcf7a0)
{
  "candidate_score": 1.0,
  "current_score": 0.0,
  "episode_failures": 0,
  "failures": {
    "fixed": 0,
    "new": 0,
    "persisting": 0
  },
  "losses": 0,
  "mutation": {
    "id": "answer-style",
    "op": "update"
  },
  "published": true,
  "selected": true,
  "selection": {
    "candidate_id": "harness-evolve-demo:harness_evolve:1:candidate",
    "evaluation": {
      "evaluator": "harness_episode_pairs",
      "evaluator_version": "1",
      "metadata": {},
      "metrics": {
        "candidate_failures": [],
        "candidate_score": 1.0,
        "candidate_scores": [
          1.0,
          0.0,
          0.0
        ],
        "current_failures": [],
        "current_score": 0.0,
        "current_scores": [
          0.0,
          0.0,
          0.0
        ],
        "episode_failures": 0
      }
    },
    "metrics": {
      "losses": 0,
      "ties": 2,
 

## Stop the service

The published tree stays under `work/`; any client can pull and install
it later through the HTTP API.

In [5]:
serve.terminate()
serve.wait(timeout=10)
print("reef stopped")


reef stopped
